# 02 - Feature Engineering

**Phase:** 2  
**Main output:** BERT embeddings, 9 behavioral features, fused raw feature matrices, labels, row IDs, and metadata for Phase 3 PCA/SVD.

This notebook is designed for Google Colab. It reads the verified Phase 1 split files and does not load or rewrite the original raw dataset.


## Runtime Contract

- Run this notebook in Google Colab.
- Use GPU runtime for BERT extraction when available.
- If Colab runs on CPU or hits OOM, reduce `BERT_BATCH_SIZE` to `4`.
- All fitted objects use train data only; validation/test are transformed with train-fitted objects.
- PCA/SVD is not performed here. That belongs to `03_PCA_Feature_Selection.ipynb`.


In [11]:
# try/except: kiểm tra notebook có đang chạy trên Google Colab hay không
try:
    # import google.colab: module chỉ tồn tại khi chạy trên Colab
    import google.colab  # type: ignore
    # IN_COLAB = True: đánh dấu môi trường Colab
    IN_COLAB = True
# except: xử lý ngoại lệ — except Exception:
except Exception:
    # IN_COLAB = False: không phải Colab
    IN_COLAB = False

# if not IN_COLAB: chặn chạy ngoài Colab theo chính sách dự án
if not IN_COLAB:
    # raise RuntimeError(: ném lỗi và dừng cell
    raise RuntimeError(
        # "Run this notebook in Google Colab unless local execution is explicitly approved...: thực thi lệnh Python
        "Run this notebook in Google Colab unless local execution is explicitly approved."
    # ): đóng ngoặc gọi hàm
    )


In [12]:
# from google.colab import drive: import API mount Google Drive trên Colab
from google.colab import drive

# drive.mount: gắn Drive để đọc/ghi artifact dự án
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# import gc: giải phóng bộ nhớ GPU/RAM sau khi trích xuất BERT
import gc
# import json: đọc/ghi metadata feature
import json
# import math: hàm toán (log, sqrt...) cho behavioral features
import math
# import random: cố định seed tái lập
import random
# import time: đo thời gian trích xuất embedding
import time
# from collections import deque: hàng đợi cho tính toán velocity/burst
from collections import deque
# from datetime import datetime, timezone: xử lý ngày giờ
from datetime import datetime, timezone
# from pathlib import Path: quản lý đường dẫn artifact
from pathlib import Path

# import joblib: lưu/tải scaler và object đã fit
import joblib
# import numpy as np: mảng số cho feature matrix
import numpy as np
# import pandas as pd: xử lý bảng split Phase 1
import pandas as pd
# from sklearn.preprocessing import StandardScaler: chuẩn hóa behavioral features
from sklearn.preprocessing import StandardScaler

# SEED = 42: hằng số seed tái lập kết quả
SEED = 42
# random.seed(SEED): cố định seed random
random.seed(SEED)
# np.random.seed(SEED): cố định seed numpy
np.random.seed(SEED)

# PROJECT_ROOT: thư mục gốc dự án trên Drive
PROJECT_ROOT = Path('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews')
# PROCESSED_DIR: dữ liệu đã split từ Phase 1
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
# FEATURE_DIR: lưu embedding và feature matrix
FEATURE_DIR = PROJECT_ROOT / 'artifacts' / 'features'
# REPORT_TABLE_DIR: bảng báo cáo Phase 2
REPORT_TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'
# REPORT_FIGURE_DIR: hình báo cáo Phase 2
REPORT_FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'

# TRAIN_PATH/VAL_PATH/TEST_PATH: file split từ Phase 1
TRAIN_PATH = PROCESSED_DIR / 'train.csv'
# VAL_PATH: biến cấu hình val path
VAL_PATH = PROCESSED_DIR / 'val.csv'
# TEST_PATH: biến cấu hình test path
TEST_PATH = PROCESSED_DIR / 'test.csv'
# SCHEMA_METADATA_PATH: metadata cột từ Phase 1
SCHEMA_METADATA_PATH = PROCESSED_DIR / 'schema_metadata.json'
# SPLIT_METADATA_PATH: metadata chia tập từ Phase 1
SPLIT_METADATA_PATH = PROCESSED_DIR / 'split_metadata.json'

# BERT_MODEL_NAME: tên model ModernBERT trên HuggingFace
BERT_MODEL_NAME = 'answerdotai/ModernBERT-base'   # ← Model mới
# MAX_LENGTH = 160: độ dài token tối đa khi encode review
MAX_LENGTH = 160                                   # ModernBERT hỗ trợ dài hơn, nhưng giữ nguyên để an toàn
# BERT_BATCH_SIZE = 8: batch size trích embedding (giảm 4 nếu OOM)
BERT_BATCH_SIZE = 8                                # Giữ 8 hoặc giảm xuống 4 nếu OOM
# FORCE_REBUILD_BERT = True: buộc tính lại embedding, không dùng cache cũ
FORCE_REBUILD_BERT = True                          # Buộc rebuild embeddings mới

# BASIC_BEHAVIORAL_FEATURES: 5 đặc trưng hành vi cơ bản
BASIC_BEHAVIORAL_FEATURES = [
    # 'basic_char_len_log',: đếm số phần tử
    'basic_char_len_log',
    # 'basic_word_count_log',: thực thi lệnh Python
    'basic_word_count_log',
    # 'basic_rating_deviation',: thực thi lệnh Python
    'basic_rating_deviation',
    # 'basic_sentiment_compound',: thực thi lệnh Python
    'basic_sentiment_compound',
    # 'basic_verified_purchase',: thực thi lệnh Python
    'basic_verified_purchase',
# ]: đóng khối danh sách
]
# ADVANCED_BEHAVIORAL_FEATURES: 4 đặc trưng hành vi nâng cao (velocity, burst...)
ADVANCED_BEHAVIORAL_FEATURES = [
    # 'adv_review_velocity_30d',: thực thi lệnh Python
    'adv_review_velocity_30d',
    # 'adv_product_burst_7d',: thực thi lệnh Python
    'adv_product_burst_7d',
    # 'adv_reviewer_behavior_score',: thực thi lệnh Python
    'adv_reviewer_behavior_score',
    # 'adv_time_gap_hours_log',: thực thi lệnh Python
    'adv_time_gap_hours_log',
# ]: đóng khối danh sách
]
# BEHAVIORAL_FEATURES: gộp 9 đặc trưng hành vi (777 = 768 BERT + 9 behavioral)
BEHAVIORAL_FEATURES = BASIC_BEHAVIORAL_FEATURES + ADVANCED_BEHAVIORAL_FEATURES

# for path: tạo thư mục output nếu chưa có
for path in [FEATURE_DIR, REPORT_TABLE_DIR, REPORT_FIGURE_DIR]:
    # path.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
    path.mkdir(parents=True, exist_ok=True)

# pd.set_option: cấu hình hiển thị DataFrame khi debug
pd.set_option('display.max_columns', 100)
# pd.set_option('display.max_colwidth', 120): cấu hình hiển thị pandas
pd.set_option('display.max_colwidth', 120)


## Load Phase 1 Splits and Schema

This section reads only `train.csv`, `val.csv`, `test.csv`, `schema_metadata.json`, and `split_metadata.json` from Phase 1.


In [14]:
# required_paths: danh sách file bắt buộc từ Phase 1
required_paths = [TRAIN_PATH, VAL_PATH, TEST_PATH, SCHEMA_METADATA_PATH, SPLIT_METADATA_PATH]
# missing_paths: lọc file chưa tồn tại trên Drive
missing_paths = [str(path) for path in required_paths if not path.exists()]
# if missing_paths: dừng nếu thiếu artifact Phase 1
if missing_paths:
    # raise FileNotFoundError(f'Missing required Phase 1 artifacts: {missing_paths}'): ném lỗi và dừng cell
    raise FileNotFoundError(f'Missing required Phase 1 artifacts: {missing_paths}')

# json.load: đọc metadata schema và split
with open(SCHEMA_METADATA_PATH, 'r', encoding='utf-8') as f:
    # schema_metadata = ...: parse nội dung JSON
    schema_metadata = json.load(f)
# with: context manager — with open(SPLIT_METADATA_PATH, 'r', encoding='utf-8') as f:
with open(SPLIT_METADATA_PATH, 'r', encoding='utf-8') as f:
    # split_metadata = ...: parse nội dung JSON
    split_metadata = json.load(f)

# pd.read_csv: nạp 3 tập train/val/test
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
# val_df = ...: đọc file CSV vào DataFrame
val_df = pd.read_csv(VAL_PATH, low_memory=False)
# test_df = ...: đọc file CSV vào DataFrame
test_df = pd.read_csv(TEST_PATH, low_memory=False)
# split_frames: dictionary gom 3 DataFrame theo tên tập
split_frames = {'train': train_df, 'val': val_df, 'test': test_df}

# schema_block: block metadata mô tả tên các cột
schema_block = schema_metadata.get('columns') or schema_metadata.get('schema') or {}


# schema_value: lấy tên cột từ metadata theo nhiều key ứng viên
def schema_value(*keys):
    # for: vòng lặp — for key in keys:
    for key in keys:
        # if: kiểm tra điều kiện — if key in schema_block and schema_block[key] is not None:
        if key in schema_block and schema_block[key] is not None:
            # return: trả kết quả từ hàm
            return schema_block[key]
        # legacy_key = ...: gán giá trị cho biến legacy key
        legacy_key = f'{key}_col'
        # if: kiểm tra điều kiện — if legacy_key in schema_block and schema_block[legacy_key] is not None
        if legacy_key in schema_block and schema_block[legacy_key] is not None:
            # return: trả kết quả từ hàm
            return schema_block[legacy_key]
    # return: trả kết quả từ hàm
    return None


# first_existing_column: tìm cột đầu tiên khớp danh sách candidates trong DataFrame
def first_existing_column(frame, candidates):
    # normalized = ...: ép kiểu chuỗi
    normalized = {str(col).lower().replace('-', '_').replace(' ', '_'): col for col in frame.columns}
    # for: vòng lặp — for candidate in candidates:
    for candidate in candidates:
        # key = ...: gán giá trị cho biến key
        key = candidate.lower().replace('-', '_').replace(' ', '_')
        # if: kiểm tra điều kiện — if key in normalized:
        if key in normalized:
            # return: trả kết quả từ hàm
            return normalized[key]
    # return: trả kết quả từ hàm
    return None


# TEXT_COL: tên cột nội dung review
TEXT_COL = schema_value('text') or first_existing_column(train_df, ['text', 'review_text', 'review', 'content'])
# LABEL_COL: tên cột nhãn nhị phân
LABEL_COL = schema_metadata.get('label_output_column') or schema_value('label_output', 'label') or 'label_binary'
# REVIEWER_COL: cột ID người đánh giá
REVIEWER_COL = schema_value('reviewer') or first_existing_column(train_df, ['user_id', 'reviewer_id', 'reviewerID'])
# PRODUCT_COL: cột ID sản phẩm
PRODUCT_COL = schema_value('product') or first_existing_column(train_df, ['asin', 'product_id', 'productID'])
# RATING_COL: cột điểm rating
RATING_COL = schema_value('rating') or first_existing_column(train_df, ['rating', 'overall', 'stars', 'score'])
# TIMESTAMP_COL: cột thời gian
TIMESTAMP_COL = schema_value('timestamp') or first_existing_column(train_df, ['timestamp', 'unixReviewTime', 'reviewTime', 'date'])
# VERIFIED_COL: cột xác nhận mua hàng (nếu có)
VERIFIED_COL = first_existing_column(train_df, ['verified_purchase', 'verified', 'is_verified_purchase'])
# HELPFUL_COL: cột vote hữu ích (nếu có)
HELPFUL_COL = first_existing_column(train_df, ['helpful_vote', 'helpful_votes', 'helpful'])

# schema_summary: tóm tắt mapping cột dùng cho Phase 2
schema_summary = {
    # 'text': TEXT_COL,: thực thi lệnh Python
    'text': TEXT_COL,
    # 'label': LABEL_COL,: thực thi lệnh Python
    'label': LABEL_COL,
    # 'reviewer': REVIEWER_COL,: thực thi lệnh Python
    'reviewer': REVIEWER_COL,
    # 'product': PRODUCT_COL,: thực thi lệnh Python
    'product': PRODUCT_COL,
    # 'rating': RATING_COL,: thực thi lệnh Python
    'rating': RATING_COL,
    # 'timestamp': TIMESTAMP_COL,: thực thi lệnh Python
    'timestamp': TIMESTAMP_COL,
    # 'verified_purchase': VERIFIED_COL,: thực thi lệnh Python
    'verified_purchase': VERIFIED_COL,
    # 'helpful_vote': HELPFUL_COL,: thực thi lệnh Python
    'helpful_vote': HELPFUL_COL,
# }: đóng khối từ điển
}

# display(pd.DataFrame([schema_summary])): hiển thị bảng/kết quả trên notebook
display(pd.DataFrame([schema_summary]))
# print('Split row counts:', {name: len(frame) for name, frame in split_frames.ite...: in thông tin ra console
print('Split row counts:', {name: len(frame) for name, frame in split_frames.items()})


,text,label,reviewer,product,rating,timestamp,verified_purchase,helpful_vote
0,text,label_binary,user_id,asin,rating,timestamp,verified_purchase,helpful_vote


Split row counts: {'train': 29923, 'val': 6413, 'test': 6413}


In [15]:
# expected_counts: số dòng kỳ vọng từ metadata Phase 1
expected_counts = split_metadata.get('row_counts', {})
# validation_rows: danh sách kết quả kiểm tra từng split
validation_rows = []

# for split_name, frame: kiểm tra từng tập train/val/test
for split_name, frame in split_frames.items():
    # expected_count = ...: gán giá trị cho biến expected count
    expected_count = expected_counts.get(split_name)
    # actual_count = ...: đếm số phần tử
    actual_count = len(frame)
    # required_missing = ...: gán giá trị cho biến required missing
    required_missing = [
        # col_name: thực thi lệnh Python
        col_name
        # for: vòng lặp — for col_name in [TEXT_COL, LABEL_COL]
        for col_name in [TEXT_COL, LABEL_COL]
        # if: kiểm tra điều kiện — if col_name is None or col_name not in frame.columns
        if col_name is None or col_name not in frame.columns
    # ]: đóng khối danh sách
    ]
    # if: kiểm tra điều kiện — if required_missing:
    if required_missing:
        # raise ValueError(f'{split_name} is missing required columns: {required_missing}'...: ném lỗi và dừng cell
        raise ValueError(f'{split_name} is missing required columns: {required_missing}')
    # if: kiểm tra điều kiện — if expected_count is not None and int(expected_count) != actual_count:
    if expected_count is not None and int(expected_count) != actual_count:
        # raise ValueError(: ném lỗi và dừng cell
        raise ValueError(
            # f'{split_name} row count mismatch: expected {expected_count}, found {actual_coun...: thực thi lệnh Python
            f'{split_name} row count mismatch: expected {expected_count}, found {actual_count}'
        # ): đóng ngoặc gọi hàm
        )

    # label_counts = ...: đếm tần suất từng giá trị
    label_counts = frame[LABEL_COL].value_counts(dropna=False).to_dict()
    # validation_rows.append({: thực thi lệnh Python
    validation_rows.append({
        # 'split': split_name,: thực thi lệnh Python
        'split': split_name,
        # 'expected_rows': expected_count,: thực thi lệnh Python
        'expected_rows': expected_count,
        # 'actual_rows': actual_count,: thực thi lệnh Python
        'actual_rows': actual_count,
        # 'missing_text': int(frame[TEXT_COL].isna().sum()),: ép kiểu số nguyên
        'missing_text': int(frame[TEXT_COL].isna().sum()),
        # 'missing_label': int(frame[LABEL_COL].isna().sum()),: ép kiểu số nguyên
        'missing_label': int(frame[LABEL_COL].isna().sum()),
        # 'label_distribution': json.dumps({str(k): int(v) for k, v in label_counts.items(...: ghi dictionary ra JSON
        'label_distribution': json.dumps({str(k): int(v) for k, v in label_counts.items()}),
        # 'status': 'pass',: thực thi lệnh Python
        'status': 'pass',
    # }): đóng từ điển hoặc DataFrame constructor
    })

# input_validation_df: DataFrame kết quả validate input Phase 1
input_validation_df = pd.DataFrame(validation_rows)
# input_validation_path: đường dẫn lưu báo cáo validate
input_validation_path = REPORT_TABLE_DIR / 'phase2_input_validation.csv'
# input_validation_df.to_csv(input_validation_path, index=False): ghi DataFrame ra file CSV
input_validation_df.to_csv(input_validation_path, index=False)
# display(input_validation_df): hiển thị bảng/kết quả trên notebook
display(input_validation_df)
# print('Saved input validation:', input_validation_path): in thông tin ra console
print('Saved input validation:', input_validation_path)


,split,expected_rows,actual_rows,missing_text,missing_label,label_distribution,status
0,train,29923,29923,0,0,"{""0"": 17677, ""1"": 12246}",pass
1,val,6413,6413,0,0,"{""0"": 3789, ""1"": 2624}",pass
2,test,6413,6413,0,0,"{""0"": 3789, ""1"": 2624}",pass


Saved input validation: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/phase2_input_validation.csv


## FEAT-01 - BERT Embeddings

This section extracts masked mean-pooled BERT embeddings by split and caches them under `artifacts/features/`. Existing caches are reused when their row count matches the split.


In [16]:
# try: import torch và transformers để trích ModernBERT embedding
try:
    # import torch: deep learning PyTorch
    import torch
    # from transformers import AutoModel, AutoTokenizer: HuggingFace transformers
    from transformers import AutoModel, AutoTokenizer
# except: xử lý ngoại lệ — except ImportError as exc:
except ImportError as exc:
    # raise ImportError(: ném lỗi và dừng cell
    raise ImportError(
        # "Install required Colab packages first, for example: !pip install transformers": transform dữ liệu bằng object đã fit
        "Install required Colab packages first, for example: !pip install transformers"
    # ) from exc: thực thi lệnh Python
    ) from exc


# mean_pool_last_hidden_state: mean pooling có mask attention → vector 768 chiều
def mean_pool_last_hidden_state(last_hidden_state, attention_mask):
    # mask = ...: ép kiểu số thực
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    # masked_embeddings = ...: gán giá trị cho biến masked embeddings
    masked_embeddings = last_hidden_state * mask
    # summed = ...: tính tổng
    summed = masked_embeddings.sum(dim=1)
    # counts = ...: tính tổng
    counts = mask.sum(dim=1).clamp(min=1e-9)
    # return: trả kết quả từ hàm
    return summed / counts


# load_bert_model: tải tokenizer + model lên GPU/CPU
def load_bert_model(model_name):
    # device = ...: chọn thiết bị GPU hoặc CPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # tokenizer = ...: gán giá trị cho biến tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # model = ...: xóa biến để giải phóng RAM/VRAM
    model = AutoModel.from_pretrained(model_name)
    # model.to(device): thực thi lệnh Python
    model.to(device)
    # model.eval(): thực thi lệnh Python
    model.eval()
    # print(f'Loaded {model_name} on {device}'): in thông tin ra console
    print(f'Loaded {model_name} on {device}')
    # if: kiểm tra điều kiện — if device.type == 'cpu':
    if device.type == 'cpu':
        # print('Warning: BERT extraction is running on CPU. It will be much slower than G...: in thông tin ra console
        print('Warning: BERT extraction is running on CPU. It will be much slower than GPU.')
    # return: trả kết quả từ hàm
    return tokenizer, model, device


# valid_cached_embedding: kiểm tra file .npy cache còn hợp lệ không
def valid_cached_embedding(path, expected_rows):
    # if: kiểm tra điều kiện — if FORCE_REBUILD_BERT or not path.exists():
    if FORCE_REBUILD_BERT or not path.exists():
        # return: trả kết quả từ hàm
        return False
    # try: bắt đầu khối thử/catch ngoại lệ
    try:
        # cached = ...: nạp mảng từ file .npy
        cached = np.load(path, mmap_mode='r')
        # return: trả kết quả từ hàm
        return cached.ndim == 2 and cached.shape[0] == expected_rows
    # except: xử lý ngoại lệ — except Exception:
    except Exception:
        # return: trả kết quả từ hàm
        return False


# extract_or_load_bert_embeddings: đọc cache hoặc trích embedding theo batch
def extract_or_load_bert_embeddings(split_name, frame, text_col, output_path, tokenizer, model, device):
    # expected_rows = ...: đếm số phần tử
    expected_rows = len(frame)
    # if: kiểm tra điều kiện — if valid_cached_embedding(output_path, expected_rows):
    if valid_cached_embedding(output_path, expected_rows):
        # cached = ...: nạp mảng từ file .npy
        cached = np.load(output_path)
        # return: ép kiểu dữ liệu cột
        return cached.astype(np.float32), {
            # 'split': split_name,: thực thi lệnh Python
            'split': split_name,
            # 'rows': int(cached.shape[0]),: ép kiểu số nguyên
            'rows': int(cached.shape[0]),
            # 'dim': int(cached.shape[1]),: ép kiểu số nguyên
            'dim': int(cached.shape[1]),
            # 'dtype': str(cached.dtype),: ép kiểu chuỗi
            'dtype': str(cached.dtype),
            # 'status': 'loaded_cache',: thực thi lệnh Python
            'status': 'loaded_cache',
            # 'elapsed_seconds': 0.0,: thực thi lệnh Python
            'elapsed_seconds': 0.0,
            # 'path': str(output_path),: ép kiểu chuỗi
            'path': str(output_path),
        # }: đóng khối từ điển
        }

    # texts = ...: điền giá trị thay thế cho NaN
    texts = frame[text_col].fillna('').astype(str).tolist()
    # batches = ...: gán giá trị cho biến batches
    batches = []
    # start_time = ...: gán giá trị cho biến start time
    start_time = time.time()
    # with: context manager — with torch.no_grad():
    with torch.no_grad():
        # for: vòng lặp — for start in range(0, len(texts), BERT_BATCH_SIZE):
        for start in range(0, len(texts), BERT_BATCH_SIZE):
            # batch_texts = ...: gán giá trị cho biến batch texts
            batch_texts = texts[start:start + BERT_BATCH_SIZE]
            # encoded = ...: gán giá trị cho biến encoded
            encoded = tokenizer(
                # batch_texts,: thực thi lệnh Python
                batch_texts,
                # padding = ...: gán giá trị cho biến padding
                padding=True,
                # truncation = ...: gán giá trị cho biến truncation
                truncation=True,
                # max_length = ...: đếm số phần tử
                max_length=MAX_LENGTH,
                # return_tensors = ...: trả kết quả từ hàm
                return_tensors='pt',
            # ): đóng ngoặc gọi hàm
            )
            # encoded = ...: gán giá trị cho biến encoded
            encoded = {key: value.to(device) for key, value in encoded.items()}
            # outputs = ...: gán giá trị cho biến outputs
            outputs = model(**encoded)
            # pooled = ...: tính trung bình
            pooled = mean_pool_last_hidden_state(outputs.last_hidden_state, encoded['attention_mask'])
            # batches.append(pooled.detach().cpu().numpy().astype(np.float32)): ép kiểu dữ liệu cột
            batches.append(pooled.detach().cpu().numpy().astype(np.float32))
            # del encoded, outputs, pooled: xóa biến để giải phóng RAM/VRAM
            del encoded, outputs, pooled
            # if: kiểm tra điều kiện — if torch.cuda.is_available():
            if torch.cuda.is_available():
                # torch.cuda.empty_cache(): thực thi lệnh Python
                torch.cuda.empty_cache()

    # embeddings = ...: ép kiểu dữ liệu cột
    embeddings = np.vstack(batches).astype(np.float32)
    # if: kiểm tra điều kiện — if embeddings.shape[0] != expected_rows:
    if embeddings.shape[0] != expected_rows:
        # raise ValueError(f'{split_name} embedding row mismatch: {embeddings.shape[0]} vs...: ném lỗi và dừng cell
        raise ValueError(f'{split_name} embedding row mismatch: {embeddings.shape[0]} vs {expected_rows}')
    # np.save(output_path, embeddings): lưu mảng numpy ra file .npy
    np.save(output_path, embeddings)
    # elapsed = ...: gán giá trị cho biến elapsed
    elapsed = time.time() - start_time
    # gc.collect(): giải phóng bộ nhớ
    gc.collect()
    # return: trả kết quả từ hàm
    return embeddings, {
        # 'split': split_name,: thực thi lệnh Python
        'split': split_name,
        # 'rows': int(embeddings.shape[0]),: ép kiểu số nguyên
        'rows': int(embeddings.shape[0]),
        # 'dim': int(embeddings.shape[1]),: ép kiểu số nguyên
        'dim': int(embeddings.shape[1]),
        # 'dtype': str(embeddings.dtype),: ép kiểu chuỗi
        'dtype': str(embeddings.dtype),
        # 'status': 'generated',: thực thi lệnh Python
        'status': 'generated',
        # 'elapsed_seconds': round(elapsed, 3),: làm tròn số
        'elapsed_seconds': round(elapsed, 3),
        # 'path': str(output_path),: ép kiểu chuỗi
        'path': str(output_path),
    # }: đóng khối từ điển
    }


In [17]:
# load_bert_model: tải ModernBERT tokenizer + model
tokenizer, bert_model, bert_device = load_bert_model(BERT_MODEL_NAME)

# bert_paths: đường dẫn lưu embedding .npy cho từng split
bert_paths = {
    # 'train': FEATURE_DIR / 'bert_train.npy',: thực thi lệnh Python
    'train': FEATURE_DIR / 'bert_train.npy',
    # 'val': FEATURE_DIR / 'bert_val.npy',: thực thi lệnh Python
    'val': FEATURE_DIR / 'bert_val.npy',
    # 'test': FEATURE_DIR / 'bert_test.npy',: thực thi lệnh Python
    'test': FEATURE_DIR / 'bert_test.npy',
# }: đóng khối từ điển
}

# extract_or_load_bert_embeddings: trích/load embedding cho train
bert_train, train_bert_report = extract_or_load_bert_embeddings(
    # 'train', train_df, TEXT_COL, bert_paths['train'], tokenizer, bert_model, bert_de...: thực thi lệnh Python
    'train', train_df, TEXT_COL, bert_paths['train'], tokenizer, bert_model, bert_device
# ): đóng ngoặc gọi hàm
)
# bert_val, val_bert_report = extract_or_load_bert_embeddings(: thực thi lệnh Python
bert_val, val_bert_report = extract_or_load_bert_embeddings(
    # 'val', val_df, TEXT_COL, bert_paths['val'], tokenizer, bert_model, bert_device: thực thi lệnh Python
    'val', val_df, TEXT_COL, bert_paths['val'], tokenizer, bert_model, bert_device
# ): đóng ngoặc gọi hàm
)
# bert_test, test_bert_report = extract_or_load_bert_embeddings(: thực thi lệnh Python
bert_test, test_bert_report = extract_or_load_bert_embeddings(
    # 'test', test_df, TEXT_COL, bert_paths['test'], tokenizer, bert_model, bert_devic...: thực thi lệnh Python
    'test', test_df, TEXT_COL, bert_paths['test'], tokenizer, bert_model, bert_device
# ): đóng ngoặc gọi hàm
)

# bert_embedding_report: bảng báo cáo quá trình trích embedding
bert_embedding_report = pd.DataFrame([train_bert_report, val_bert_report, test_bert_report])
# bert_embedding_report['model_name'] = BERT_MODEL_NAME: thực thi lệnh Python
bert_embedding_report['model_name'] = BERT_MODEL_NAME
# bert_embedding_report['max_length'] = MAX_LENGTH: đếm số phần tử
bert_embedding_report['max_length'] = MAX_LENGTH
# bert_embedding_report['batch_size'] = BERT_BATCH_SIZE: thực thi lệnh Python
bert_embedding_report['batch_size'] = BERT_BATCH_SIZE
# bert_embedding_report_path: lưu báo cáo embedding ra CSV
bert_embedding_report_path = REPORT_TABLE_DIR / 'phase2_bert_embedding_report.csv'
# bert_embedding_report.to_csv(bert_embedding_report_path, index=False): ghi DataFrame ra file CSV
bert_embedding_report.to_csv(bert_embedding_report_path, index=False)
# display(bert_embedding_report): hiển thị bảng/kết quả trên notebook
display(bert_embedding_report)

# del bert_model + empty_cache: giải phóng VRAM sau khi trích xong
del bert_model
# if: kiểm tra điều kiện — if torch.cuda.is_available():
if torch.cuda.is_available():
    # torch.cuda.empty_cache(): thực thi lệnh Python
    torch.cuda.empty_cache()
# gc.collect(): giải phóng bộ nhớ
gc.collect()
# print('Saved BERT embedding report:', bert_embedding_report_path): in thông tin ra console
print('Saved BERT embedding report:', bert_embedding_report_path)


,split,rows,dim,dtype,status,elapsed_seconds,path,model_name,max_length,batch_size
0,train,29923,768,float32,generated,299.748,/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/bert_train.npy,answerdotai/ModernBERT-base,160,8
1,val,6413,768,float32,generated,62.628,/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/bert_val.npy,answerdotai/ModernBERT-base,160,8
2,test,6413,768,float32,generated,65.102,/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/bert_test.npy,answerdotai/ModernBERT-base,160,8


Saved BERT embedding report: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/phase2_bert_embedding_report.csv


## FEAT-02, FEAT-03, FEAT-05 - Behavioral Features and Leakage Checks

The notebook creates exactly 9 behavioral features: 5 basic and 4 advanced. No labels are used to construct behavioral feature values.


In [18]:
# behavioral_metadata_notes: ghi chú fallback khi thiếu cột phụ trợ
behavioral_metadata_notes = []


# parse_review_datetime: chuyển cột thời gian sang datetime (unix hoặc chuỗi)
def parse_review_datetime(series):
    # numeric = ...: gán giá trị cho biến numeric
    numeric = pd.to_numeric(series, errors='coerce')
    # if: kiểm tra điều kiện — if numeric.notna().mean() > 0.9:
    if numeric.notna().mean() > 0.9:
        # median_value = ...: xóa dòng/cột có giá trị thiếu
        median_value = numeric.dropna().median()
        # if: kiểm tra điều kiện — if median_value > 1e12:
        if median_value > 1e12:
            # return: trả kết quả từ hàm
            return pd.to_datetime(numeric, unit='ms', errors='coerce')
        # if: kiểm tra điều kiện — if median_value > 1e9:
        if median_value > 1e9:
            # return: trả kết quả từ hàm
            return pd.to_datetime(numeric, unit='s', errors='coerce')
    # return: trả kết quả từ hàm
    return pd.to_datetime(series, errors='coerce')


# load_sentiment_scorer: thử VADER → NLTK VADER → TextBlob để tính sentiment
def load_sentiment_scorer():
    # try: bắt đầu khối thử/catch ngoại lệ
    try:
        # from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer: import thư viện vaderSentiment
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        # analyzer = ...: gán giá trị cho biến analyzer
        analyzer = SentimentIntensityAnalyzer()
        # return: trả kết quả từ hàm
        return 'vaderSentiment', lambda text: analyzer.polarity_scores(text)['compound']
    # except: xử lý ngoại lệ — except Exception as vader_error:
    except Exception as vader_error:
        # try: bắt đầu khối thử/catch ngoại lệ
        try:
            # import nltk: import thư viện nltk
            import nltk
            # from nltk.sentiment import SentimentIntensityAnalyzer: import thư viện nltk
            from nltk.sentiment import SentimentIntensityAnalyzer
            # try: bắt đầu khối thử/catch ngoại lệ
            try:
                # nltk.data.find('sentiment/vader_lexicon.zip'): ghép song song nhiều iterable
                nltk.data.find('sentiment/vader_lexicon.zip')
            # except: xử lý ngoại lệ — except LookupError:
            except LookupError:
                # nltk.download('vader_lexicon', quiet=True): thực thi lệnh Python
                nltk.download('vader_lexicon', quiet=True)
            # analyzer = ...: gán giá trị cho biến analyzer
            analyzer = SentimentIntensityAnalyzer()
            # return: trả kết quả từ hàm
            return 'nltk_vader', lambda text: analyzer.polarity_scores(text)['compound']
        # except: xử lý ngoại lệ — except Exception as nltk_error:
        except Exception as nltk_error:
            # try: bắt đầu khối thử/catch ngoại lệ
            try:
                # from textblob import TextBlob: import thư viện textblob
                from textblob import TextBlob
                # return: trả kết quả từ hàm
                return 'TextBlob', lambda text: TextBlob(text).sentiment.polarity
            # except: xử lý ngoại lệ — except Exception as textblob_error:
            except Exception as textblob_error:
                # behavioral_metadata_notes.append(: thực thi lệnh Python
                behavioral_metadata_notes.append(
                    # 'Sentiment unavailable; filled basic_sentiment_compound with 0. ': thực thi lệnh Python
                    'Sentiment unavailable; filled basic_sentiment_compound with 0. '
                    # f'Errors: {vader_error}; {nltk_error}; {textblob_error}': thực thi lệnh Python
                    f'Errors: {vader_error}; {nltk_error}; {textblob_error}'
                # ): đóng ngoặc gọi hàm
                )
                # return: trả kết quả từ hàm
                return 'unavailable', lambda text: 0.0


# SENTIMENT_BACKEND, sentiment_score_fn: backend và hàm tính điểm sentiment
SENTIMENT_BACKEND, sentiment_score_fn = load_sentiment_scorer()
# print('Sentiment backend:', SENTIMENT_BACKEND): in thông tin ra console
print('Sentiment backend:', SENTIMENT_BACKEND)


Sentiment backend: nltk_vader


In [19]:
# train_rating_mean: trung bình rating trên train (dùng impute cho val/test)
train_rating_mean = (
    # pd.to_numeric(train_df[RATING_COL], errors='coerce').mean(): tính trung bình
    pd.to_numeric(train_df[RATING_COL], errors='coerce').mean()
    # if: kiểm tra điều kiện — if RATING_COL is not None and RATING_COL in train_df.columns
    if RATING_COL is not None and RATING_COL in train_df.columns
    # else 0.0: thực thi lệnh Python
    else 0.0
# ): đóng ngoặc gọi hàm
)
# if: kiểm tra điều kiện — if pd.isna(train_rating_mean):
if pd.isna(train_rating_mean):
    # train_rating_mean = ...: tính trung bình
    train_rating_mean = 0.0
    # behavioral_metadata_notes.append('Rating mean unavailable; basic_rating_deviatio...: tính trung bình
    behavioral_metadata_notes.append('Rating mean unavailable; basic_rating_deviation fallback used.')


# add_basic_behavioral_features: tính 5 đặc trưng hành vi cơ bản
def add_basic_behavioral_features(frame):
    # result = ...: gán giá trị cho biến result
    result = pd.DataFrame(index=frame.index)
    # text_series = ...: điền giá trị thay thế cho NaN
    text_series = frame[TEXT_COL].fillna('').astype(str)
    # result['basic_char_len_log'] = np.log1p(text_series.str.len()): đếm số phần tử
    result['basic_char_len_log'] = np.log1p(text_series.str.len())
    # result['basic_word_count_log'] = np.log1p(text_series.str.split().str.len()): đếm số phần tử
    result['basic_word_count_log'] = np.log1p(text_series.str.split().str.len())

    # if: kiểm tra điều kiện — if RATING_COL is not None and RATING_COL in frame.columns:
    if RATING_COL is not None and RATING_COL in frame.columns:
        # rating_values = ...: điền giá trị thay thế cho NaN
        rating_values = pd.to_numeric(frame[RATING_COL], errors='coerce').fillna(train_rating_mean)
        # result['basic_rating_deviation'] = (rating_values - train_rating_mean).abs(): tính trung bình
        result['basic_rating_deviation'] = (rating_values - train_rating_mean).abs()
    # else: nhánh còn lại của điều kiện
    else:
        # result['basic_rating_deviation'] = 0.0: thực thi lệnh Python
        result['basic_rating_deviation'] = 0.0

    # result['basic_sentiment_compound'] = text_series.map(sentiment_score_fn).astype(...: ép kiểu dữ liệu cột
    result['basic_sentiment_compound'] = text_series.map(sentiment_score_fn).astype(float)

    # if: kiểm tra điều kiện — if VERIFIED_COL is not None and VERIFIED_COL in frame.columns:
    if VERIFIED_COL is not None and VERIFIED_COL in frame.columns:
        # verified = ...: gán giá trị cho biến verified
        verified = frame[VERIFIED_COL]
        # if: kiểm tra điều kiện — if verified.dtype == bool:
        if verified.dtype == bool:
            # result['basic_verified_purchase'] = verified.astype(float): ép kiểu dữ liệu cột
            result['basic_verified_purchase'] = verified.astype(float)
        # else: nhánh còn lại của điều kiện
        else:
            # result['basic_verified_purchase'] = (: thực thi lệnh Python
            result['basic_verified_purchase'] = (
                # verified.astype(str).str.strip().str.lower().isin(['true', '1', 'yes', 'y']): ép kiểu dữ liệu cột
                verified.astype(str).str.strip().str.lower().isin(['true', '1', 'yes', 'y'])
            # ).astype(float): ép kiểu dữ liệu cột
            ).astype(float)
    # else: nhánh còn lại của điều kiện
    else:
        # result['basic_verified_purchase'] = 0.0: thực thi lệnh Python
        result['basic_verified_purchase'] = 0.0
    # return: trả kết quả từ hàm
    return result


# basic_features: dictionary 5 đặc trưng cơ bản cho train/val/test
basic_features = {
    # split_name: add_basic_behavioral_features(frame): thực thi lệnh Python
    split_name: add_basic_behavioral_features(frame)
    # for: vòng lặp — for split_name, frame in split_frames.items()
    for split_name, frame in split_frames.items()
# }: đóng khối từ điển
}
# display(basic_features['train'].head()): hiển thị bảng/kết quả trên notebook
display(basic_features['train'].head())


,basic_char_len_log,basic_word_count_log,basic_rating_deviation,basic_sentiment_compound,basic_verified_purchase
0,4.691348,2.995732,1.064566,0.4404,1.0
1,6.333280,4.779123,1.064566,0.9918,1.0
2,5.843544,4.330733,1.064566,0.9252,1.0
3,3.637586,1.791759,2.935434,0.0000,1.0
4,4.700480,2.995732,1.064566,0.0258,1.0


In [20]:
# prior_window_count_for_targets: đếm số review trong cửa sổ ngày trước mỗi dòng target
def prior_window_count_for_targets(target_frame, history_frame, group_col, datetime_col, window_days, include_target_history):
    # if: kiểm tra điều kiện — if group_col is None or datetime_col is None or group_col not in targe
    if group_col is None or datetime_col is None or group_col not in target_frame.columns:
        # return: trả kết quả từ hàm
        return pd.Series(0.0, index=target_frame.index)
    # if: kiểm tra điều kiện — if datetime_col not in target_frame.columns:
    if datetime_col not in target_frame.columns:
        # return: trả kết quả từ hàm
        return pd.Series(0.0, index=target_frame.index)

    # target = ...: gán giá trị cho biến target
    target = pd.DataFrame({
        # '_group': target_frame[group_col].astype(str).values,: ép kiểu dữ liệu cột
        '_group': target_frame[group_col].astype(str).values,
        # '_time': parse_review_datetime(target_frame[datetime_col]).values,: thực thi lệnh Python
        '_time': parse_review_datetime(target_frame[datetime_col]).values,
        # '_target': True,: thực thi lệnh Python
        '_target': True,
        # '_target_index': target_frame.index.values,: thực thi lệnh Python
        '_target_index': target_frame.index.values,
    # }): đóng từ điển hoặc DataFrame constructor
    })
    # history = ...: gán giá trị cho biến history
    history = pd.DataFrame({
        # '_group': history_frame[group_col].astype(str).values,: ép kiểu dữ liệu cột
        '_group': history_frame[group_col].astype(str).values,
        # '_time': parse_review_datetime(history_frame[datetime_col]).values,: thực thi lệnh Python
        '_time': parse_review_datetime(history_frame[datetime_col]).values,
        # '_target': False,: thực thi lệnh Python
        '_target': False,
        # '_target_index': -1,: thực thi lệnh Python
        '_target_index': -1,
    # }): đóng từ điển hoặc DataFrame constructor
    })
    # combined = ...: nối nhiều DataFrame
    combined = pd.concat([history, target], ignore_index=True)
    # combined = ...: xóa dòng/cột có giá trị thiếu
    combined = combined.dropna(subset=['_time']).sort_values(['_group', '_time', '_target'])

    # counts = ...: gán giá trị cho biến counts
    counts = pd.Series(0.0, index=target_frame.index)
    # window = ...: gán giá trị cho biến window
    window = pd.Timedelta(days=window_days)
    # for: vòng lặp — for _, group_data in combined.groupby('_group', sort=False):
    for _, group_data in combined.groupby('_group', sort=False):
        # seen_times = ...: gán giá trị cho biến seen times
        seen_times = deque()
        # for: vòng lặp — for _, row in group_data.iterrows():
        for _, row in group_data.iterrows():
            # current_time = ...: gán giá trị cho biến current time
            current_time = row['_time']
            # while: lặp khi điều kiện đúng — while seen_times and current_time - seen_times[0] > window:
            while seen_times and current_time - seen_times[0] > window:
                # seen_times.popleft(): thực thi lệnh Python
                seen_times.popleft()
            # if: kiểm tra điều kiện — if bool(row['_target']):
            if bool(row['_target']):
                # counts.loc[int(row['_target_index'])] = float(len(seen_times)): ép kiểu số thực
                counts.loc[int(row['_target_index'])] = float(len(seen_times))
                # if: kiểm tra điều kiện — if include_target_history:
                if include_target_history:
                    # seen_times.append(current_time): thực thi lệnh Python
                    seen_times.append(current_time)
            # else: nhánh còn lại của điều kiện
            else:
                # seen_times.append(current_time): thực thi lệnh Python
                seen_times.append(current_time)
    # return: trả kết quả từ hàm
    return counts


# previous_time_gap_hours_for_targets: số giờ từ review trước của cùng reviewer
def previous_time_gap_hours_for_targets(target_frame, history_frame, group_col, datetime_col, include_target_history):
    # if: kiểm tra điều kiện — if group_col is None or datetime_col is None or group_col not in targe
    if group_col is None or datetime_col is None or group_col not in target_frame.columns:
        # return: trả kết quả từ hàm
        return pd.Series(np.nan, index=target_frame.index)
    # if: kiểm tra điều kiện — if datetime_col not in target_frame.columns:
    if datetime_col not in target_frame.columns:
        # return: trả kết quả từ hàm
        return pd.Series(np.nan, index=target_frame.index)

    # target = ...: gán giá trị cho biến target
    target = pd.DataFrame({
        # '_group': target_frame[group_col].astype(str).values,: ép kiểu dữ liệu cột
        '_group': target_frame[group_col].astype(str).values,
        # '_time': parse_review_datetime(target_frame[datetime_col]).values,: thực thi lệnh Python
        '_time': parse_review_datetime(target_frame[datetime_col]).values,
        # '_target': True,: thực thi lệnh Python
        '_target': True,
        # '_target_index': target_frame.index.values,: thực thi lệnh Python
        '_target_index': target_frame.index.values,
    # }): đóng từ điển hoặc DataFrame constructor
    })
    # history = ...: gán giá trị cho biến history
    history = pd.DataFrame({
        # '_group': history_frame[group_col].astype(str).values,: ép kiểu dữ liệu cột
        '_group': history_frame[group_col].astype(str).values,
        # '_time': parse_review_datetime(history_frame[datetime_col]).values,: thực thi lệnh Python
        '_time': parse_review_datetime(history_frame[datetime_col]).values,
        # '_target': False,: thực thi lệnh Python
        '_target': False,
        # '_target_index': -1,: thực thi lệnh Python
        '_target_index': -1,
    # }): đóng từ điển hoặc DataFrame constructor
    })
    # combined = ...: nối nhiều DataFrame
    combined = pd.concat([history, target], ignore_index=True)
    # combined = ...: xóa dòng/cột có giá trị thiếu
    combined = combined.dropna(subset=['_time']).sort_values(['_group', '_time', '_target'])

    # gaps = ...: gán giá trị cho biến gaps
    gaps = pd.Series(np.nan, index=target_frame.index)
    # for: vòng lặp — for _, group_data in combined.groupby('_group', sort=False):
    for _, group_data in combined.groupby('_group', sort=False):
        # previous_time = ...: gán giá trị cho biến previous time
        previous_time = None
        # for: vòng lặp — for _, row in group_data.iterrows():
        for _, row in group_data.iterrows():
            # current_time = ...: gán giá trị cho biến current time
            current_time = row['_time']
            # if: kiểm tra điều kiện — if bool(row['_target']):
            if bool(row['_target']):
                # if: kiểm tra điều kiện — if previous_time is not None:
                if previous_time is not None:
                    # gaps.loc[int(row['_target_index'])] = (current_time - previous_time).total_secon...: ép kiểu số nguyên
                    gaps.loc[int(row['_target_index'])] = (current_time - previous_time).total_seconds() / 3600.0
                # if: kiểm tra điều kiện — if include_target_history:
                if include_target_history:
                    # previous_time = ...: gán giá trị cho biến previous time
                    previous_time = current_time
            # else: nhánh còn lại của điều kiện
            else:
                # previous_time = ...: gán giá trị cho biến previous time
                previous_time = current_time
    # return: trả kết quả từ hàm
    return gaps


# fit_reviewer_behavior_map: fit điểm hành vi reviewer chỉ trên train (không dùng label)
def fit_reviewer_behavior_map(train_frame, train_basic_features):
    # if: kiểm tra điều kiện — if REVIEWER_COL is None or REVIEWER_COL not in train_frame.columns:
    if REVIEWER_COL is None or REVIEWER_COL not in train_frame.columns:
        # return: trả kết quả từ hàm
        return {}, 0.0, {'status': 'skipped', 'reason': 'reviewer column missing'}

    # reviewer_source = ...: gán giá trị cho biến reviewer source
    reviewer_source = pd.DataFrame({
        # 'reviewer_id': train_frame[REVIEWER_COL].astype(str),: ép kiểu dữ liệu cột
        'reviewer_id': train_frame[REVIEWER_COL].astype(str),
        # 'review_count': 1.0,: thực thi lệnh Python
        'review_count': 1.0,
        # 'rating_deviation': train_basic_features['basic_rating_deviation'].values,: thực thi lệnh Python
        'rating_deviation': train_basic_features['basic_rating_deviation'].values,
        # 'char_len_log': train_basic_features['basic_char_len_log'].values,: đếm số phần tử
        'char_len_log': train_basic_features['basic_char_len_log'].values,
        # 'sentiment': train_basic_features['basic_sentiment_compound'].values,: thực thi lệnh Python
        'sentiment': train_basic_features['basic_sentiment_compound'].values,
    # }): đóng từ điển hoặc DataFrame constructor
    })
    # summary = ...: nhóm dữ liệu theo cột
    summary = reviewer_source.groupby('reviewer_id').agg(
        # reviewer_review_count = ...: tính tổng
        reviewer_review_count=('review_count', 'sum'),
        # reviewer_mean_rating_deviation = ...: tính trung bình
        reviewer_mean_rating_deviation=('rating_deviation', 'mean'),
        # reviewer_mean_char_len_log = ...: tính trung bình
        reviewer_mean_char_len_log=('char_len_log', 'mean'),
        # reviewer_mean_sentiment = ...: tính trung bình
        reviewer_mean_sentiment=('sentiment', 'mean'),
    # ): đóng ngoặc gọi hàm
    )
    # score_columns = ...: gán giá trị cho biến score columns
    score_columns = [
        # 'reviewer_review_count',: thực thi lệnh Python
        'reviewer_review_count',
        # 'reviewer_mean_rating_deviation',: tính trung bình
        'reviewer_mean_rating_deviation',
        # 'reviewer_mean_char_len_log',: tính trung bình
        'reviewer_mean_char_len_log',
        # 'reviewer_mean_sentiment',: tính trung bình
        'reviewer_mean_sentiment',
    # ]: đóng khối danh sách
    ]
    # score_matrix = ...: tính tổng
    score_matrix = summary[score_columns].copy()
    # means = ...: tính trung bình
    means = score_matrix.mean()
    # stds = ...: gán giá trị cho biến stds
    stds = score_matrix.std(ddof=0).replace(0, 1.0)
    # summary['adv_reviewer_behavior_score'] = ((score_matrix - means) / stds).mean(ax...: tính trung bình
    summary['adv_reviewer_behavior_score'] = ((score_matrix - means) / stds).mean(axis=1)
    # global_score = ...: ép kiểu số thực
    global_score = float(summary['adv_reviewer_behavior_score'].mean())
    # return: trả kết quả từ hàm
    return summary['adv_reviewer_behavior_score'].to_dict(), global_score, {
        # 'status': 'generated',: thực thi lệnh Python
        'status': 'generated',
        # 'reviewer_count': int(summary.shape[0]),: ép kiểu số nguyên
        'reviewer_count': int(summary.shape[0]),
        # 'global_score': global_score,: thực thi lệnh Python
        'global_score': global_score,
    # }: đóng khối từ điển
    }


# reviewer_behavior_map: ánh xạ reviewer_id → behavior score (fit train only)
reviewer_behavior_map, reviewer_behavior_global_score, reviewer_behavior_meta = fit_reviewer_behavior_map(
    # train_df, basic_features['train']: thực thi lệnh Python
    train_df, basic_features['train']
# ): đóng ngoặc gọi hàm
)


In [21]:
# timestamp_parse_rates: tỷ lệ parse thành công cột thời gian theo split
timestamp_parse_rates = {}
# for: vòng lặp — for split_name, frame in split_frames.items():
for split_name, frame in split_frames.items():
    # if: kiểm tra điều kiện — if TIMESTAMP_COL is not None and TIMESTAMP_COL in frame.columns:
    if TIMESTAMP_COL is not None and TIMESTAMP_COL in frame.columns:
        # timestamp_parse_rates[split_name] = float(parse_review_datetime(frame[TIMESTAMP_...: ép kiểu số thực
        timestamp_parse_rates[split_name] = float(parse_review_datetime(frame[TIMESTAMP_COL]).notna().mean())
    # else: nhánh còn lại của điều kiện
    else:
        # timestamp_parse_rates[split_name] = 0.0: thực thi lệnh Python
        timestamp_parse_rates[split_name] = 0.0

# train_gap_raw = ...: gán giá trị cho biến train gap raw
train_gap_raw = previous_time_gap_hours_for_targets(
    # train_df, train_df.iloc[0:0], REVIEWER_COL, TIMESTAMP_COL, include_target_histor...: thực thi lệnh Python
    train_df, train_df.iloc[0:0], REVIEWER_COL, TIMESTAMP_COL, include_target_history=True
# ): đóng ngoặc gọi hàm
)
# train_gap_median = ...: xóa dòng/cột có giá trị thiếu
train_gap_median = train_gap_raw.replace([np.inf, -np.inf], np.nan).dropna().median()
# if: kiểm tra điều kiện — if pd.isna(train_gap_median):
if pd.isna(train_gap_median):
    # train_gap_median = ...: tính trung vị
    train_gap_median = 0.0


# add_advanced_behavioral_features: tính 4 đặc trưng hành vi nâng cao (chỉ dùng history train)
def add_advanced_behavioral_features(split_name, frame):
    # is_train = ...: gán giá trị cho biến is train
    is_train = split_name == 'train'
    # history_frame = ...: gán giá trị cho biến history frame
    history_frame = train_df.iloc[0:0] if is_train else train_df
    # result = ...: gán giá trị cho biến result
    result = pd.DataFrame(index=frame.index)
    # result['adv_review_velocity_30d'] = prior_window_count_for_targets(: thực thi lệnh Python
    result['adv_review_velocity_30d'] = prior_window_count_for_targets(
        # frame, history_frame, REVIEWER_COL, TIMESTAMP_COL, 30, include_target_history=is...: thực thi lệnh Python
        frame, history_frame, REVIEWER_COL, TIMESTAMP_COL, 30, include_target_history=is_train
    # ): đóng ngoặc gọi hàm
    )
    # result['adv_product_burst_7d'] = prior_window_count_for_targets(: thực thi lệnh Python
    result['adv_product_burst_7d'] = prior_window_count_for_targets(
        # frame, history_frame, PRODUCT_COL, TIMESTAMP_COL, 7, include_target_history=is_t...: thực thi lệnh Python
        frame, history_frame, PRODUCT_COL, TIMESTAMP_COL, 7, include_target_history=is_train
    # ): đóng ngoặc gọi hàm
    )
    # if: kiểm tra điều kiện — if REVIEWER_COL is not None and REVIEWER_COL in frame.columns and revi
    if REVIEWER_COL is not None and REVIEWER_COL in frame.columns and reviewer_behavior_map:
        # result['adv_reviewer_behavior_score'] = (: thực thi lệnh Python
        result['adv_reviewer_behavior_score'] = (
            # frame[REVIEWER_COL].astype(str).map(reviewer_behavior_map).fillna(reviewer_behav...: điền giá trị thay thế cho NaN
            frame[REVIEWER_COL].astype(str).map(reviewer_behavior_map).fillna(reviewer_behavior_global_score)
        # ): đóng ngoặc gọi hàm
        )
    # else: nhánh còn lại của điều kiện
    else:
        # result['adv_reviewer_behavior_score'] = reviewer_behavior_global_score: thực thi lệnh Python
        result['adv_reviewer_behavior_score'] = reviewer_behavior_global_score

    # gaps = ...: gán giá trị cho biến gaps
    gaps = previous_time_gap_hours_for_targets(
        # frame, history_frame, REVIEWER_COL, TIMESTAMP_COL, include_target_history=is_tra...: thực thi lệnh Python
        frame, history_frame, REVIEWER_COL, TIMESTAMP_COL, include_target_history=is_train
    # ): đóng ngoặc gọi hàm
    )
    # gaps = ...: điền giá trị thay thế cho NaN
    gaps = gaps.replace([np.inf, -np.inf], np.nan).fillna(train_gap_median).clip(lower=0)
    # result['adv_time_gap_hours_log'] = np.log1p(gaps): thực thi lệnh Python
    result['adv_time_gap_hours_log'] = np.log1p(gaps)
    # return: trả kết quả từ hàm
    return result


# advanced_features: dictionary 4 đặc trưng nâng cao cho từng split
advanced_features = {
    # split_name: add_advanced_behavioral_features(split_name, frame): thực thi lệnh Python
    split_name: add_advanced_behavioral_features(split_name, frame)
    # for: vòng lặp — for split_name, frame in split_frames.items()
    for split_name, frame in split_frames.items()
# }: đóng khối từ điển
}

# behavioral_features: gộp 9 cột behavioral (5 basic + 4 advanced)
behavioral_features = {}
# for: vòng lặp — for split_name in split_frames:
for split_name in split_frames:
    # behavioral_features[split_name] = pd.concat(: nối nhiều DataFrame
    behavioral_features[split_name] = pd.concat(
        # [basic_features[split_name], advanced_features[split_name]],: thực thi lệnh Python
        [basic_features[split_name], advanced_features[split_name]],
        # axis = ...: gán giá trị cho biến axis
        axis=1,
    # )[BEHAVIORAL_FEATURES].astype(np.float32): ép kiểu dữ liệu cột
    )[BEHAVIORAL_FEATURES].astype(np.float32)
    # if: kiểm tra điều kiện — if list(behavioral_features[split_name].columns) != BEHAVIORAL_FEATURE
    if list(behavioral_features[split_name].columns) != BEHAVIORAL_FEATURES:
        # raise ValueError(f'{split_name} behavioral feature column order mismatch'): ném lỗi và dừng cell
        raise ValueError(f'{split_name} behavioral feature column order mismatch')
    # if: kiểm tra điều kiện — if len(BEHAVIORAL_FEATURES) != 9:
    if len(BEHAVIORAL_FEATURES) != 9:
        # raise ValueError('Expected exactly 9 behavioral feature columns'): ném lỗi và dừng cell
        raise ValueError('Expected exactly 9 behavioral feature columns')

# display(behavioral_features['train'].head()): hiển thị bảng/kết quả trên notebook
display(behavioral_features['train'].head())


,basic_char_len_log,basic_word_count_log,basic_rating_deviation,basic_sentiment_compound,basic_verified_purchase,adv_review_velocity_30d,adv_product_burst_7d,adv_reviewer_behavior_score,adv_time_gap_hours_log
0,4.691348,2.995732,1.064566,0.4404,1.0,0.0,0.0,-0.078962,8.157623
1,6.333280,4.779123,1.064566,0.9918,1.0,0.0,0.0,0.586466,8.157623
2,5.843544,4.330733,1.064566,0.9252,1.0,0.0,0.0,0.438494,8.157623
3,3.637586,1.791759,2.935434,0.0000,1.0,0.0,0.0,0.032443,8.157623
4,4.700480,2.995732,1.064566,0.0258,1.0,0.0,0.0,-0.290797,8.157623


In [22]:
# behavioral_paths: đường dẫn lưu CSV behavioral features
behavioral_paths = {
    # 'train': FEATURE_DIR / 'behavioral_train.csv',: thực thi lệnh Python
    'train': FEATURE_DIR / 'behavioral_train.csv',
    # 'val': FEATURE_DIR / 'behavioral_val.csv',: thực thi lệnh Python
    'val': FEATURE_DIR / 'behavioral_val.csv',
    # 'test': FEATURE_DIR / 'behavioral_test.csv',: thực thi lệnh Python
    'test': FEATURE_DIR / 'behavioral_test.csv',
# }: đóng khối từ điển
}

# for: xuất behavioral features kèm row_id ra CSV
for split_name, feature_frame in behavioral_features.items():
    # output_frame = ...: gán giá trị cho biến output frame
    output_frame = feature_frame.copy()
    # output_frame.insert(0, 'row_id', np.arange(len(output_frame))): tạo dãy số cho vòng lặp
    output_frame.insert(0, 'row_id', np.arange(len(output_frame)))
    # output_frame.to_csv(behavioral_paths[split_name], index=False): ghi DataFrame ra file CSV
    output_frame.to_csv(behavioral_paths[split_name], index=False)

# feature_dictionary_rows: mô tả công thức và chính sách fit từng feature
feature_dictionary_rows = [
    # {'feature_name': 'basic_char_len_log', 'feature_group': 'basic', 'formula': 'log...: fit model/reducer trên dữ liệu train
    {'feature_name': 'basic_char_len_log', 'feature_group': 'basic', 'formula': 'log1p(character length of review text)', 'source_columns': TEXT_COL, 'fit_policy': 'deterministic per row', 'missing_fallback': 'empty text -> 0 length'},
    # {'feature_name': 'basic_word_count_log', 'feature_group': 'basic', 'formula': 'l...: fit model/reducer trên dữ liệu train
    {'feature_name': 'basic_word_count_log', 'feature_group': 'basic', 'formula': 'log1p(word count of review text)', 'source_columns': TEXT_COL, 'fit_policy': 'deterministic per row', 'missing_fallback': 'empty text -> 0 words'},
    # {'feature_name': 'basic_rating_deviation', 'feature_group': 'basic', 'formula': ...: tính trung bình
    {'feature_name': 'basic_rating_deviation', 'feature_group': 'basic', 'formula': 'abs(rating - train_rating_mean)', 'source_columns': RATING_COL, 'fit_policy': 'train rating mean only', 'missing_fallback': 'train mean or 0 if rating missing'},
    # {'feature_name': 'basic_sentiment_compound', 'feature_group': 'basic', 'formula'...: fit model/reducer trên dữ liệu train
    {'feature_name': 'basic_sentiment_compound', 'feature_group': 'basic', 'formula': 'VADER compound or TextBlob polarity', 'source_columns': TEXT_COL, 'fit_policy': 'external lexicon, no label fitting', 'missing_fallback': '0 if backend unavailable'},
    # {'feature_name': 'basic_verified_purchase', 'feature_group': 'basic', 'formula':...: fit model/reducer trên dữ liệu train
    {'feature_name': 'basic_verified_purchase', 'feature_group': 'basic', 'formula': 'binary verified purchase flag', 'source_columns': VERIFIED_COL, 'fit_policy': 'deterministic per row', 'missing_fallback': '0 if column missing'},
    # {'feature_name': 'adv_review_velocity_30d', 'feature_group': 'advanced', 'formul...: fit model/reducer trên dữ liệu train
    {'feature_name': 'adv_review_velocity_30d', 'feature_group': 'advanced', 'formula': 'prior reviewer reviews in 30-day chronological window', 'source_columns': f'{REVIEWER_COL}, {TIMESTAMP_COL}', 'fit_policy': 'train rows use prior train rows; val/test use train history only', 'missing_fallback': '0 if reviewer/time missing'},
    # {'feature_name': 'adv_product_burst_7d', 'feature_group': 'advanced', 'formula':...: fit model/reducer trên dữ liệu train
    {'feature_name': 'adv_product_burst_7d', 'feature_group': 'advanced', 'formula': 'prior product reviews in 7-day chronological window', 'source_columns': f'{PRODUCT_COL}, {TIMESTAMP_COL}', 'fit_policy': 'train rows use prior train rows; val/test use train history only', 'missing_fallback': '0 if product/time missing'},
    # {'feature_name': 'adv_reviewer_behavior_score', 'feature_group': 'advanced', 'fo...: fit model/reducer trên dữ liệu train
    {'feature_name': 'adv_reviewer_behavior_score', 'feature_group': 'advanced', 'formula': 'train-fitted unsupervised reviewer summary score', 'source_columns': f'{REVIEWER_COL}, rating/text/sentiment derived columns', 'fit_policy': 'reviewer map fit on train only, no labels', 'missing_fallback': 'train global score for unknown reviewer'},
    # {'feature_name': 'adv_time_gap_hours_log', 'feature_group': 'advanced', 'formula...: tính trung vị
    {'feature_name': 'adv_time_gap_hours_log', 'feature_group': 'advanced', 'formula': 'log1p(hours since previous reviewer review)', 'source_columns': f'{REVIEWER_COL}, {TIMESTAMP_COL}', 'fit_policy': 'train rows use prior train rows; val/test use train history only', 'missing_fallback': 'train median gap or 0'},
# ]: đóng khối danh sách
]
# feature_dictionary = ...: tạo dictionary
feature_dictionary = pd.DataFrame(feature_dictionary_rows)
# feature_dictionary_path = ...: tạo dictionary
feature_dictionary_path = FEATURE_DIR / 'feature_dictionary.csv'
# feature_dictionary.to_csv(feature_dictionary_path, index=False): ghi DataFrame ra file CSV
feature_dictionary.to_csv(feature_dictionary_path, index=False)

# leakage_rows: kiểm tra không rò rỉ nhãn/val vào feature engineering
leakage_rows = [
    # {'check': 'behavioral_feature_count', 'status': 'pass', 'detail': str(len(BEHAVI...: đếm số phần tử
    {'check': 'behavioral_feature_count', 'status': 'pass', 'detail': str(len(BEHAVIORAL_FEATURES))},
    # {'check': 'label_not_used_in_feature_formulas', 'status': 'pass', 'detail': 'No ...: thực thi lệnh Python
    {'check': 'label_not_used_in_feature_formulas', 'status': 'pass', 'detail': 'No behavioral feature formula uses labels.'},
    # {'check': 'rating_deviation_fit_policy', 'status': 'pass', 'detail': 'Uses train...: tính trung bình
    {'check': 'rating_deviation_fit_policy', 'status': 'pass', 'detail': 'Uses train_rating_mean only.'},
    # {'check': 'reviewer_behavior_map_fit_policy', 'status': 'pass', 'detail': json.d...: ghi dictionary ra JSON
    {'check': 'reviewer_behavior_map_fit_policy', 'status': 'pass', 'detail': json.dumps(reviewer_behavior_meta)},
    # {'check': 'timestamp_parse_rates', 'status': 'pass', 'detail': json.dumps(timest...: ghi dictionary ra JSON
    {'check': 'timestamp_parse_rates', 'status': 'pass', 'detail': json.dumps(timestamp_parse_rates)},
    # {'check': 'sentiment_backend', 'status': 'pass' if SENTIMENT_BACKEND != 'unavail...: thực thi lệnh Python
    {'check': 'sentiment_backend', 'status': 'pass' if SENTIMENT_BACKEND != 'unavailable' else 'warning', 'detail': SENTIMENT_BACKEND},
    # {'check': 'scaler_not_fit_before_fusion_section', 'status': 'pass', 'detail': 'S...: fit model/reducer trên dữ liệu train
    {'check': 'scaler_not_fit_before_fusion_section', 'status': 'pass', 'detail': 'Scaler fit occurs only in the fusion section.'},
# ]: đóng khối danh sách
]
# if: kiểm tra điều kiện — if REVIEWER_COL is not None and REVIEWER_COL in train_df.columns and r
if REVIEWER_COL is not None and REVIEWER_COL in train_df.columns and reviewer_behavior_map:
    # train_reviewers = ...: ép kiểu dữ liệu cột
    train_reviewers = set(train_df[REVIEWER_COL].astype(str))
    # for: vòng lặp — for split_name, frame in [('val', val_df), ('test', test_df)]:
    for split_name, frame in [('val', val_df), ('test', test_df)]:
        # unknown_count = ...: ép kiểu dữ liệu cột
        unknown_count = int((~frame[REVIEWER_COL].astype(str).isin(train_reviewers)).sum())
        # leakage_rows.append({'check': f'{split_name}_unknown_reviewer_fallback_count', '...: ép kiểu chuỗi
        leakage_rows.append({'check': f'{split_name}_unknown_reviewer_fallback_count', 'status': 'pass', 'detail': str(unknown_count)})

# leakage_check_report = ...: gán giá trị cho biến leakage check report
leakage_check_report = pd.DataFrame(leakage_rows)
# leakage_check_report_path = ...: gán giá trị cho biến leakage check report path
leakage_check_report_path = REPORT_TABLE_DIR / 'phase2_leakage_check_report.csv'
# leakage_check_report.to_csv(leakage_check_report_path, index=False): ghi DataFrame ra file CSV
leakage_check_report.to_csv(leakage_check_report_path, index=False)

# display(feature_dictionary): hiển thị bảng/kết quả trên notebook
display(feature_dictionary)
# display(leakage_check_report): hiển thị bảng/kết quả trên notebook
display(leakage_check_report)
# print('Saved behavioral feature files:', behavioral_paths): in thông tin ra console
print('Saved behavioral feature files:', behavioral_paths)
# print('Saved feature dictionary:', feature_dictionary_path): in thông tin ra console
print('Saved feature dictionary:', feature_dictionary_path)
# print('Saved leakage check report:', leakage_check_report_path): in thông tin ra console
print('Saved leakage check report:', leakage_check_report_path)


,feature_name,feature_group,formula,source_columns,fit_policy,missing_fallback
0,basic_char_len_log,basic,log1p(character length of review text),text,deterministic per row,empty text -> 0 length
1,basic_word_count_log,basic,log1p(word count of review text),text,deterministic per row,empty text -> 0 words
2,basic_rating_deviation,basic,abs(rating - train_rating_mean),rating,train rating mean only,train mean or 0 if rating missing
3,basic_sentiment_compound,basic,VADER compound or TextBlob polarity,text,"external lexicon, no label fitting",0 if backend unavailable
4,basic_verified_purchase,basic,binary verified purchase flag,verified_purchase,deterministic per row,0 if column missing
5,adv_review_velocity_30d,advanced,prior reviewer reviews in 30-day chronological window,"user_id, timestamp",train rows use prior train rows; val/test use train history only,0 if reviewer/time missing
6,adv_product_burst_7d,advanced,prior product reviews in 7-day chronological window,"asin, timestamp",train rows use prior train rows; val/test use train history only,0 if product/time missing
7,adv_reviewer_behavior_score,advanced,train-fitted unsupervised reviewer summary score,"user_id, rating/text/sentiment derived columns","reviewer map fit on train only, no labels",train global score for unknown reviewer
8,adv_time_gap_hours_log,advanced,log1p(hours since previous reviewer review),"user_id, timestamp",train rows use prior train rows; val/test use train history only,train median gap or 0


,check,status,detail
0,behavioral_feature_count,pass,9
1,label_not_used_in_feature_formulas,pass,No behavioral feature formula uses labels.
2,rating_deviation_fit_policy,pass,Uses train_rating_mean only.
3,reviewer_behavior_map_fit_policy,pass,"{""status"": ""generated"", ""reviewer_count"": 29628, ""global_score"": -7.29056945021839e-17}"
4,timestamp_parse_rates,pass,"{""train"": 1.0, ""val"": 1.0, ""test"": 1.0}"
5,sentiment_backend,pass,nltk_vader
6,scaler_not_fit_before_fusion_section,pass,Scaler fit occurs only in the fusion section.
7,val_unknown_reviewer_fallback_count,pass,6299
8,test_unknown_reviewer_fallback_count,pass,6307


Saved behavioral feature files: {'train': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/behavioral_train.csv'), 'val': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/behavioral_val.csv'), 'test': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/behavioral_test.csv')}
Saved feature dictionary: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/feature_dictionary.csv
Saved leakage check report: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/phase2_leakage_check_report.csv


## FEAT-04 and FEAT-05 - Train-Only Scaling, Fusion, Metadata

Behavioral features are scaled with `StandardScaler` fit on train only, then concatenated with BERT embeddings.


In [23]:
# behavioral_scaler: StandardScaler fit CHỈ trên train
behavioral_scaler = StandardScaler()
# fit_transform train, transform val/test (tránh leakage)
behavioral_train_scaled = behavioral_scaler.fit_transform(behavioral_features['train'][BEHAVIORAL_FEATURES])
# behavioral_val_scaled = ...: transform dữ liệu bằng object đã fit
behavioral_val_scaled = behavioral_scaler.transform(behavioral_features['val'][BEHAVIORAL_FEATURES])
# behavioral_test_scaled = ...: transform dữ liệu bằng object đã fit
behavioral_test_scaled = behavioral_scaler.transform(behavioral_features['test'][BEHAVIORAL_FEATURES])

# scaled_behavioral = ...: gán giá trị cho biến scaled behavioral
scaled_behavioral = {
    # 'train': behavioral_train_scaled.astype(np.float32),: ép kiểu dữ liệu cột
    'train': behavioral_train_scaled.astype(np.float32),
    # 'val': behavioral_val_scaled.astype(np.float32),: ép kiểu dữ liệu cột
    'val': behavioral_val_scaled.astype(np.float32),
    # 'test': behavioral_test_scaled.astype(np.float32),: ép kiểu dữ liệu cột
    'test': behavioral_test_scaled.astype(np.float32),
# }: đóng khối từ điển
}

# for: vòng lặp — for split_name, array in scaled_behavioral.items():
for split_name, array in scaled_behavioral.items():
    # if: kiểm tra điều kiện — if not np.isfinite(array).all():
    if not np.isfinite(array).all():
        # raise ValueError(f'Non-finite scaled behavioral values in {split_name}'): ném lỗi và dừng cell
        raise ValueError(f'Non-finite scaled behavioral values in {split_name}')
    # if: kiểm tra điều kiện — if array.shape[0] != len(split_frames[split_name]):
    if array.shape[0] != len(split_frames[split_name]):
        # raise ValueError(f'{split_name} scaled behavioral row mismatch'): ném lỗi và dừng cell
        raise ValueError(f'{split_name} scaled behavioral row mismatch')

# behavioral_scaler_path = ...: gán giá trị cho biến behavioral scaler path
behavioral_scaler_path = FEATURE_DIR / 'behavioral_scaler.joblib'
# joblib.dump(behavioral_scaler, behavioral_scaler_path): lưu object (scaler/model) ra disk
joblib.dump(behavioral_scaler, behavioral_scaler_path)

# bert_arrays = ...: gán giá trị cho biến bert arrays
bert_arrays = {'train': bert_train, 'val': bert_val, 'test': bert_test}
# features_raw: nối BERT (768) + behavioral scaled (9) → 777 chiều
features_raw = {}
# for: vòng lặp — for split_name in ['train', 'val', 'test']:
for split_name in ['train', 'val', 'test']:
    # bert_array = ...: ép kiểu dữ liệu cột
    bert_array = bert_arrays[split_name].astype(np.float32)
    # behavioral_array = ...: gán giá trị cho biến behavioral array
    behavioral_array = scaled_behavioral[split_name]
    # if: kiểm tra điều kiện — if bert_array.shape[0] != behavioral_array.shape[0]:
    if bert_array.shape[0] != behavioral_array.shape[0]:
        # raise ValueError(f'{split_name} BERT/behavioral row mismatch'): ném lỗi và dừng cell
        raise ValueError(f'{split_name} BERT/behavioral row mismatch')
    # fused = ...: nối các mảng numpy
    fused = np.concatenate([bert_array, behavioral_array], axis=1).astype(np.float32)
    # expected_dim = ...: đếm số phần tử
    expected_dim = bert_array.shape[1] + len(BEHAVIORAL_FEATURES)
    # if: kiểm tra điều kiện — if fused.shape[1] != expected_dim:
    if fused.shape[1] != expected_dim:
        # raise ValueError(f'{split_name} fused dimension mismatch: {fused.shape[1]} vs {e...: ném lỗi và dừng cell
        raise ValueError(f'{split_name} fused dimension mismatch: {fused.shape[1]} vs {expected_dim}')
    # features_raw[split_name] = fused: thực thi lệnh Python
    features_raw[split_name] = fused

# feature_paths = ...: gán giá trị cho biến feature paths
feature_paths = {'train': FEATURE_DIR / 'features_raw_train.npy', 'val': FEATURE_DIR / 'features_raw_val.npy', 'test': FEATURE_DIR / 'features_raw_test.npy'}
# label_paths = ...: gán giá trị cho biến label paths
label_paths = {'train': FEATURE_DIR / 'labels_train.npy', 'val': FEATURE_DIR / 'labels_val.npy', 'test': FEATURE_DIR / 'labels_test.npy'}
# row_id_paths = ...: gán giá trị cho biến row id paths
row_id_paths = {'train': FEATURE_DIR / 'row_ids_train.csv', 'val': FEATURE_DIR / 'row_ids_val.csv', 'test': FEATURE_DIR / 'row_ids_test.csv'}

# for: vòng lặp — for split_name, frame in split_frames.items():
for split_name, frame in split_frames.items():
    # np.save(feature_paths[split_name], features_raw[split_name]): lưu mảng numpy ra file .npy
    np.save(feature_paths[split_name], features_raw[split_name])
    # labels = ...: ép kiểu dữ liệu cột
    labels = frame[LABEL_COL].astype(int).to_numpy(dtype=np.int64)
    # np.save(label_paths[split_name], labels): lưu mảng numpy ra file .npy
    np.save(label_paths[split_name], labels)
    # row_id_df = ...: tạo dãy số cho vòng lặp
    row_id_df = pd.DataFrame({'row_id': np.arange(len(frame))})
    # for: vòng lặp — for col_name in [REVIEWER_COL, PRODUCT_COL, TIMESTAMP_COL]:
    for col_name in [REVIEWER_COL, PRODUCT_COL, TIMESTAMP_COL]:
        # if: kiểm tra điều kiện — if col_name is not None and col_name in frame.columns:
        if col_name is not None and col_name in frame.columns:
            # row_id_df[col_name] = frame[col_name].values: thực thi lệnh Python
            row_id_df[col_name] = frame[col_name].values
    # row_id_df.to_csv(row_id_paths[split_name], index=False): ghi DataFrame ra file CSV
    row_id_df.to_csv(row_id_paths[split_name], index=False)

# print('Saved fused feature arrays:', feature_paths): in thông tin ra console
print('Saved fused feature arrays:', feature_paths)
# print('Saved labels:', label_paths): in thông tin ra console
print('Saved labels:', label_paths)
# print('Saved row IDs:', row_id_paths): in thông tin ra console
print('Saved row IDs:', row_id_paths)


Saved fused feature arrays: {'train': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/features_raw_train.npy'), 'val': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/features_raw_val.npy'), 'test': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/features_raw_test.npy')}
Saved labels: {'train': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/labels_train.npy'), 'val': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/labels_val.npy'), 'test': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/labels_test.npy')}
Saved row IDs: {'train': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/row_ids_train.csv'), 'val': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/row_ids_val.csv'), 'test': PosixPath('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/arti

In [24]:
# array_memory_mb: tính dung lượng RAM (MB) của numpy array
def array_memory_mb(array):
    # return: trả kết quả từ hàm
    return round(float(array.nbytes) / (1024 ** 2), 4)


# feature_quality_rows = ...: gán giá trị cho biến feature quality rows
feature_quality_rows = []
# memory_rows = ...: gán giá trị cho biến memory rows
memory_rows = []
# for: vòng lặp — for split_name in ['train', 'val', 'test']:
for split_name in ['train', 'val', 'test']:
    # raw = ...: gán giá trị cho biến raw
    raw = features_raw[split_name]
    # behavior = ...: gán giá trị cho biến behavior
    behavior = behavioral_features[split_name][BEHAVIORAL_FEATURES]
    # labels = ...: ép kiểu dữ liệu cột
    labels = split_frames[split_name][LABEL_COL].astype(int)
    # feature_quality_rows.append({: thực thi lệnh Python
    feature_quality_rows.append({
        # 'split': split_name,: thực thi lệnh Python
        'split': split_name,
        # 'rows': int(raw.shape[0]),: ép kiểu số nguyên
        'rows': int(raw.shape[0]),
        # 'bert_dim': int(bert_arrays[split_name].shape[1]),: ép kiểu số nguyên
        'bert_dim': int(bert_arrays[split_name].shape[1]),
        # 'behavioral_dim': len(BEHAVIORAL_FEATURES),: đếm số phần tử
        'behavioral_dim': len(BEHAVIORAL_FEATURES),
        # 'fused_dim': int(raw.shape[1]),: ép kiểu số nguyên
        'fused_dim': int(raw.shape[1]),
        # 'missing_behavioral_values': int(behavior.isna().sum().sum()),: ép kiểu số nguyên
        'missing_behavioral_values': int(behavior.isna().sum().sum()),
        # 'non_finite_fused_values': int((~np.isfinite(raw)).sum()),: kiểm tra phần tử hữu hạn
        'non_finite_fused_values': int((~np.isfinite(raw)).sum()),
        # 'label_0_count': int((labels == 0).sum()),: ép kiểu số nguyên
        'label_0_count': int((labels == 0).sum()),
        # 'label_1_count': int((labels == 1).sum()),: ép kiểu số nguyên
        'label_1_count': int((labels == 1).sum()),
    # }): đóng từ điển hoặc DataFrame constructor
    })
    # memory_rows.extend([: thực thi lệnh Python
    memory_rows.extend([
        # {'split': split_name, 'artifact': f'bert_{split_name}.npy', 'shape': str(tuple(b...: ép kiểu chuỗi
        {'split': split_name, 'artifact': f'bert_{split_name}.npy', 'shape': str(tuple(bert_arrays[split_name].shape)), 'dtype': str(bert_arrays[split_name].dtype), 'memory_mb': array_memory_mb(bert_arrays[split_name])},
        # {'split': split_name, 'artifact': f'features_raw_{split_name}.npy', 'shape': str...: ép kiểu chuỗi
        {'split': split_name, 'artifact': f'features_raw_{split_name}.npy', 'shape': str(tuple(raw.shape)), 'dtype': str(raw.dtype), 'memory_mb': array_memory_mb(raw)},
    # ]): đóng list comprehension hoặc danh sách
    ])

# feature_quality_report = ...: gán giá trị cho biến feature quality report
feature_quality_report = pd.DataFrame(feature_quality_rows)
# feature_quality_report_path = ...: gán giá trị cho biến feature quality report path
feature_quality_report_path = REPORT_TABLE_DIR / 'phase2_feature_quality_report.csv'
# feature_quality_report.to_csv(feature_quality_report_path, index=False): ghi DataFrame ra file CSV
feature_quality_report.to_csv(feature_quality_report_path, index=False)

# memory_report = ...: gán giá trị cho biến memory report
memory_report = pd.DataFrame(memory_rows)
# memory_report_path = ...: gán giá trị cho biến memory report path
memory_report_path = REPORT_TABLE_DIR / 'phase2_memory_report.csv'
# memory_report.to_csv(memory_report_path, index=False): ghi DataFrame ra file CSV
memory_report.to_csv(memory_report_path, index=False)

# feature_metadata: JSON metadata đầy đủ cho Phase 3 trở đi
feature_metadata = {
    # 'generated_at_utc': datetime.now(timezone.utc).isoformat(),: thực thi lệnh Python
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    # 'seed': SEED,: thực thi lệnh Python
    'seed': SEED,
    # 'bert': {: thực thi lệnh Python
    'bert': {
        # 'model_name': BERT_MODEL_NAME,: thực thi lệnh Python
        'model_name': BERT_MODEL_NAME,
        # 'pooling': 'masked_mean_last_hidden_state',: tính trung bình
        'pooling': 'masked_mean_last_hidden_state',
        # 'max_length': MAX_LENGTH,: đếm số phần tử
        'max_length': MAX_LENGTH,
        # 'batch_size': BERT_BATCH_SIZE,: thực thi lệnh Python
        'batch_size': BERT_BATCH_SIZE,
        # 'embedding_dim': int(bert_arrays['train'].shape[1]),: ép kiểu số nguyên
        'embedding_dim': int(bert_arrays['train'].shape[1]),
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
    # 'behavioral_features': {: thực thi lệnh Python
    'behavioral_features': {
        # 'all': BEHAVIORAL_FEATURES,: thực thi lệnh Python
        'all': BEHAVIORAL_FEATURES,
        # 'basic': BASIC_BEHAVIORAL_FEATURES,: thực thi lệnh Python
        'basic': BASIC_BEHAVIORAL_FEATURES,
        # 'advanced': ADVANCED_BEHAVIORAL_FEATURES,: thực thi lệnh Python
        'advanced': ADVANCED_BEHAVIORAL_FEATURES,
        # 'count': len(BEHAVIORAL_FEATURES),: đếm số phần tử
        'count': len(BEHAVIORAL_FEATURES),
        # 'sentiment_backend': SENTIMENT_BACKEND,: thực thi lệnh Python
        'sentiment_backend': SENTIMENT_BACKEND,
        # 'train_rating_mean': float(train_rating_mean),: ép kiểu số thực
        'train_rating_mean': float(train_rating_mean),
        # 'reviewer_behavior_meta': reviewer_behavior_meta,: thực thi lệnh Python
        'reviewer_behavior_meta': reviewer_behavior_meta,
        # 'notes': behavioral_metadata_notes,: thực thi lệnh Python
        'notes': behavioral_metadata_notes,
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
    # 'fusion': {: thực thi lệnh Python
    'fusion': {
        # 'order': ['bert_embedding', 'scaled_behavioral_features'],: thực thi lệnh Python
        'order': ['bert_embedding', 'scaled_behavioral_features'],
        # 'fused_dim': int(features_raw['train'].shape[1]),: ép kiểu số nguyên
        'fused_dim': int(features_raw['train'].shape[1]),
        # 'behavioral_scaler_path': str(behavioral_scaler_path),: ép kiểu chuỗi
        'behavioral_scaler_path': str(behavioral_scaler_path),
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
    # 'schema': schema_summary,: tính tổng
    'schema': schema_summary,
    # 'row_counts': {split_name: int(len(frame)) for split_name, frame in split_frames...: đếm số phần tử
    'row_counts': {split_name: int(len(frame)) for split_name, frame in split_frames.items()},
    # 'artifacts': {: thực thi lệnh Python
    'artifacts': {
        # 'bert': {split_name: str(path) for split_name, path in bert_paths.items()},: ép kiểu chuỗi
        'bert': {split_name: str(path) for split_name, path in bert_paths.items()},
        # 'behavioral': {split_name: str(path) for split_name, path in behavioral_paths.it...: ép kiểu chuỗi
        'behavioral': {split_name: str(path) for split_name, path in behavioral_paths.items()},
        # 'features_raw': {split_name: str(path) for split_name, path in feature_paths.ite...: ép kiểu chuỗi
        'features_raw': {split_name: str(path) for split_name, path in feature_paths.items()},
        # 'labels': {split_name: str(path) for split_name, path in label_paths.items()},: ép kiểu chuỗi
        'labels': {split_name: str(path) for split_name, path in label_paths.items()},
        # 'row_ids': {split_name: str(path) for split_name, path in row_id_paths.items()},: ép kiểu chuỗi
        'row_ids': {split_name: str(path) for split_name, path in row_id_paths.items()},
        # 'feature_dictionary': str(feature_dictionary_path),: tạo dictionary
        'feature_dictionary': str(feature_dictionary_path),
        # 'feature_quality_report': str(feature_quality_report_path),: ép kiểu chuỗi
        'feature_quality_report': str(feature_quality_report_path),
        # 'memory_report': str(memory_report_path),: ép kiểu chuỗi
        'memory_report': str(memory_report_path),
        # 'leakage_check_report': str(leakage_check_report_path),: ép kiểu chuỗi
        'leakage_check_report': str(leakage_check_report_path),
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
    # 'leakage_policy': {: thực thi lệnh Python
    'leakage_policy': {
        # 'scaler_fit': 'train only',: fit model/reducer trên dữ liệu train
        'scaler_fit': 'train only',
        # 'rating_mean': 'train only',: tính trung bình
        'rating_mean': 'train only',
        # 'reviewer_behavior_map': 'fit on train only without labels',: fit model/reducer trên dữ liệu train
        'reviewer_behavior_map': 'fit on train only without labels',
        # 'validation_test_transform': 'use train-fitted scaler and train-fitted aggregate...: transform dữ liệu bằng object đã fit
        'validation_test_transform': 'use train-fitted scaler and train-fitted aggregate maps',
        # 'label_usage': 'labels are saved but not used to construct feature values',: ép kiểu chuỗi
        'label_usage': 'labels are saved but not used to construct feature values',
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
# }: đóng khối từ điển
}
# feature_metadata_path = ...: gán giá trị cho biến feature metadata path
feature_metadata_path = FEATURE_DIR / 'feature_metadata.json'
# with: context manager — with open(feature_metadata_path, 'w', encoding='utf-8') as f:
with open(feature_metadata_path, 'w', encoding='utf-8') as f:
    # json.dump(feature_metadata, f, indent=2, ensure_ascii=False): ghi dictionary ra JSON
    json.dump(feature_metadata, f, indent=2, ensure_ascii=False)

# display(feature_quality_report): hiển thị bảng/kết quả trên notebook
display(feature_quality_report)
# display(memory_report): hiển thị bảng/kết quả trên notebook
display(memory_report)
# print('Saved feature metadata:', feature_metadata_path): in thông tin ra console
print('Saved feature metadata:', feature_metadata_path)


,split,rows,bert_dim,behavioral_dim,fused_dim,missing_behavioral_values,non_finite_fused_values,label_0_count,label_1_count
0,train,29923,768,9,777,0,0,17677,12246
1,val,6413,768,9,777,0,0,3789,2624
2,test,6413,768,9,777,0,0,3789,2624


,split,artifact,shape,dtype,memory_mb
0,train,bert_train.npy,"(29923, 768)",float32,87.6650
1,train,features_raw_train.npy,"(29923, 777)",float32,88.6924
2,val,bert_val.npy,"(6413, 768)",float32,18.7881
3,val,features_raw_val.npy,"(6413, 777)",float32,19.0083
4,test,bert_test.npy,"(6413, 768)",float32,18.7881
5,test,features_raw_test.npy,"(6413, 777)",float32,19.0083


Saved feature metadata: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/features/feature_metadata.json


## Phase 2 Completion Checklist

- **FEAT-01:** BERT embeddings are extracted by split and cached as `bert_train.npy`, `bert_val.npy`, `bert_test.npy`.
- **FEAT-02:** Five basic behavioral features are exported with documented formulas.
- **FEAT-03:** Four advanced behavioral features are exported: review velocity, product burst, reviewer behavior score, and time gap.
- **FEAT-04:** Behavioral features are scaled with train-only `StandardScaler` and fused with BERT embeddings.
- **FEAT-05:** Leakage checks are exported; labels are not used to construct feature values.

Phase 3 must read:

- `artifacts/features/features_raw_train.npy`
- `artifacts/features/features_raw_val.npy`
- `artifacts/features/features_raw_test.npy`
- `artifacts/features/labels_train.npy`
- `artifacts/features/labels_val.npy`
- `artifacts/features/labels_test.npy`
- `artifacts/features/feature_metadata.json`
- `artifacts/features/behavioral_scaler.joblib`
- `artifacts/features/feature_dictionary.csv`
- `reports/tables/phase2_feature_quality_report.csv`
- `reports/tables/phase2_memory_report.csv`
- `reports/tables/phase2_leakage_check_report.csv`
